# Beginner Tutorial 3: How Distributed Computing Works

## What You Will Learn

- The client-scheduler-worker architecture
- Vertical vs. horizontal scaling
- Concurrency vs. parallelism
- How providers abstract execution backends
- Manual vs. adaptive scaling strategies
- Amdahl's Law (when more workers don't help)

## Prerequisites

- Completed notebooks 01–02
- `pip install scalable`

In [ ]:
import os
import tempfile
import time

project_dir = tempfile.mkdtemp(prefix="scalable-beginner-03-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## 💡 Key Concept: The Scheduler-Worker Architecture

Distributed systems have three roles:

```
┌──────────┐         ┌────────────┐         ┌──────────┐
│  Client  │────────▶│  Scheduler │────────▶│ Worker 1 │
│ (you)    │         │ (Dask)     │────────▶│ Worker 2 │
│          │◀────────│            │────────▶│ Worker 3 │
└──────────┘         └────────────┘         └──────────┘
   submit()            assigns tasks          executes &
   gather()            tracks state           returns results
```

- **Client** = your script (submits work, collects results)
- **Scheduler** = traffic controller (decides where tasks run)
- **Workers** = labor force (actually execute functions)

## 💡 Key Concept: Vertical vs. Horizontal Scaling

- **Vertical (scale UP):** Bigger machine (more CPUs, more RAM)
- **Horizontal (scale OUT):** More machines working together

Scalable focuses on horizontal scaling — distributing work across many workers.

In [ ]:
# Write manifest for scaling experiments
manifest_content = """\
version: 1
project:
  name: scaling-demo

targets:
  local:
    provider: local
    max_workers: 4
    threads_per_worker: 1
    processes: false
    containers: none

components:
  analysis:
    cpus: 1
    memory: 1G

tasks:
  run_analysis:
    component: analysis
"""

with open("scalable.yaml", "w") as f:
    f.write(manifest_content)
print("Manifest ready with 4 workers")

## Step 1: Demonstrating Parallelism

Let's see the speedup from distributing work across workers.

In [ ]:
from scalable import ScalableSession

def simulate_work(task_id: int) -> dict:
    """Each task takes 0.5 seconds."""
    time.sleep(0.5)
    return {"task_id": task_id, "result": task_id * 10}

# First: run SEQUENTIALLY (no Scalable) as baseline
start = time.time()
sequential_results = [simulate_work(i) for i in range(8)]
sequential_time = time.time() - start
print(f"Sequential (1 worker): {sequential_time:.2f}s for 8 tasks")

In [ ]:
# Now: run in PARALLEL with Scalable (4 workers)
session = ScalableSession.from_manifest("./scalable.yaml", target="local")

start = time.time()
futures = [session.submit(simulate_work, i, task="run_analysis") for i in range(8)]
parallel_results = session.gather(futures)
parallel_time = time.time() - start

print(f"Parallel (4 workers): {parallel_time:.2f}s for 8 tasks")
print(f"Speedup: {sequential_time / parallel_time:.1f}x")
print(f"\n💡 With 4 workers and 8 tasks: 2 batches × 0.5s ≈ 1.0s (ideal)")
print(f"   Actual overhead brings it slightly above ideal.")

session.close()

## 🤔 Think About It: Amdahl's Law

**Amdahl's Law** says: speedup is limited by the sequential portion.

If 90% of work can be parallelized and 10% is sequential:
- 10 workers → ~5.3× speedup (not 10×)
- 100 workers → ~9.2× speedup (not 100×)

**Lesson:** Don't throw infinite workers at a problem. There's always a point
of diminishing returns. Telemetry helps you find the sweet spot.

## 💡 Key Concept: Provider Pattern

Providers are **abstraction layers** — they hide complexity behind a simple interface.

Like a light switch: you don't need to know if your electricity comes from
solar, nuclear, or gas. The switch (provider API) works the same regardless.

| Provider | What it does | When to use |
|----------|-------------|-------------|
| `local` | Threads/processes on your machine | Development |
| `slurm` | Jobs on HPC cluster | Production batch |
| `aws` | Containers on AWS Fargate | Cloud burst |
| `kubernetes` | Pods in K8s cluster | Shared infrastructure |

## 📖 Vocabulary Summary

| Term | Definition |
|------|------------|
| Horizontal Scaling | Adding more machines/workers |
| Vertical Scaling | Getting a bigger single machine |
| Scheduler | Assigns tasks to available workers |
| Worker | Process/thread that executes tasks |
| Client | Your script (submits work, gathers results) |
| Concurrency | Multiple tasks in progress (maybe not simultaneous) |
| Parallelism | Multiple tasks executing at exact same instant |
| Provider | Abstraction over execution backend |
| Adaptive Scaling | Auto-adjusting worker count based on demand |
| Amdahl's Law | Speedup limited by sequential portion |

In [ ]:
# Cleanup
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print(f"Cleaned up {project_dir}")